1.
<p>Generate 3,000 fictional UPI-style transactions in a notebook named 'upi_dates.ipynb' with seed 42: txn_id, txn_time as text 'DD/MM/YYYY HH:MM' between 01/01/2025 and 30/06/2025, amount_inr 10 to 5000, and category (Grocery, Fuel, Recharge, Food, Bills). Save 'upi_raw.csv', reload it and convert txn_time with `pd.to_datetime(format='%d/%m/%Y %H:%M')`, printing `dtypes`. Hint: Build times as a start date plus `pd.to_timedelta(minutes, unit='m')`, then `.dt.strftime`.</p>
2.
<p>Extract `year`, `month`, `day`, `weekday` (day name), `quarter` and `hour` from txn_time with the `.dt` accessor. Create `is_weekend`, `is_month_end` and `time_slot` (Morning 5 to 11, Afternoon 12 to 16, Evening 17 to 21, Night otherwise) as new time-based features, and print the transaction count per `time_slot`. Constraint: Build `time_slot` with `.loc` masks, not `pd.cut`.</p>
3.
<p>Set txn_time as a sorted index, then use `resample` to build `daily_total` ('D'), `weekly_total` ('W') and `monthly_total` (month-end) sums of amount_inr. Assert the three grand totals are equal and save them as 'upi_daily.csv', 'upi_weekly.csv' and 'upi_monthly.csv'. Hint: Month-end is 'ME' in recent pandas and 'M' in older versions.</p>
4.
<p>Treat txn_time as UTC server time: apply `dt.tz_localize('UTC')` then `dt.tz_convert('Asia/Kolkata')` to make `txn_time_ist`. Print the first row before and after (the shift must be 5 hours 30 minutes), compare the busiest hour before and after conversion, and save 'upi_clean.csv'. Run Restart & Run All and push to 's09_upi_dates' in your GitHub repo 'ds_analysis_portfolio'.</p>

In [2]:
import numpy as np
import pandas as pd

In [3]:
np.random.seed(42)

# Generate transaction IDs
txn_id = range(1, 3001)

# Start and end dates
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-06-30 23:59")

# Total minutes between dates
total_minutes = int((end_date - start_date).total_seconds() / 60)

# Generate random minutes
random_minutes = np.random.randint(
    0, total_minutes + 1, size=3000
)

# Generate transaction times
txn_times = start_date + pd.to_timedelta(
    random_minutes, unit="m"
)

# Convert dates into text format
txn_time = txn_times.strftime("%d/%m/%Y %H:%M")

# Generate random amounts
amount_inr = np.random.randint(10, 5001, size=3000)

# Generate categories
categories = ["Grocery", "Fuel", "Recharge", "Food", "Bills"]

category = np.random.choice(categories, size=3000)

# Create DataFrame
upi = pd.DataFrame({
    "txn_id": txn_id,
    "txn_time": txn_time,
    "amount_inr": amount_inr,
    "category": category
})

print(upi.head())
print("Total transactions:", len(upi))

   txn_id          txn_time  amount_inr  category
0       1  26/03/2025 16:38        2840  Recharge
1       2  12/04/2025 23:47          52  Recharge
2       3  02/04/2025 14:52         953  Recharge
3       4  14/03/2025 00:14        2400     Bills
4       5  29/06/2025 23:38        2555     Bills
Total transactions: 3000


In [4]:
# Save raw data
upi.to_csv("upi_raw.csv", index=False)

# Reload CSV
upi = pd.read_csv("upi_raw.csv")

# Convert text into datetime
upi["txn_time"] = pd.to_datetime(
    upi["txn_time"],
    format="%d/%m/%Y %H:%M"
)

print(upi.head())

print("\nData Types:")
print(upi.dtypes)

   txn_id            txn_time  amount_inr  category
0       1 2025-03-26 16:38:00        2840  Recharge
1       2 2025-04-12 23:47:00          52  Recharge
2       3 2025-04-02 14:52:00         953  Recharge
3       4 2025-03-14 00:14:00        2400     Bills
4       5 2025-06-29 23:38:00        2555     Bills

Data Types:
txn_id                 int64
txn_time      datetime64[us]
amount_inr             int64
category                 str
dtype: object


In [5]:
upi["year"] = upi["txn_time"].dt.year

upi["month"] = upi["txn_time"].dt.month

upi["day"] = upi["txn_time"].dt.day

upi["weekday"] = upi["txn_time"].dt.day_name()

upi["quarter"] = upi["txn_time"].dt.quarter

upi["hour"] = upi["txn_time"].dt.hour

print(upi.head())

   txn_id            txn_time  amount_inr  category  year  month  day  \
0       1 2025-03-26 16:38:00        2840  Recharge  2025      3   26   
1       2 2025-04-12 23:47:00          52  Recharge  2025      4   12   
2       3 2025-04-02 14:52:00         953  Recharge  2025      4    2   
3       4 2025-03-14 00:14:00        2400     Bills  2025      3   14   
4       5 2025-06-29 23:38:00        2555     Bills  2025      6   29   

     weekday  quarter  hour  
0  Wednesday        1    16  
1   Saturday        2    23  
2  Wednesday        2    14  
3     Friday        1     0  
4     Sunday        2    23  


In [6]:
# Check whether transaction happened on weekend
upi["is_weekend"] = upi["txn_time"].dt.dayofweek >= 5

# Check whether transaction happened on the last day of month
upi["is_month_end"] = upi["txn_time"].dt.is_month_end

print(upi[[
    "txn_time",
    "weekday",
    "is_weekend",
    "is_month_end"
]].head(10))

             txn_time    weekday  is_weekend  is_month_end
0 2025-03-26 16:38:00  Wednesday       False         False
1 2025-04-12 23:47:00   Saturday        True         False
2 2025-04-02 14:52:00  Wednesday       False         False
3 2025-03-14 00:14:00     Friday       False         False
4 2025-06-29 23:38:00     Sunday        True         False
5 2025-03-25 05:59:00    Tuesday       False         False
6 2025-03-18 13:48:00    Tuesday       False         False
7 2025-05-25 08:52:00     Sunday        True         False
8 2025-02-08 02:46:00   Saturday        True         False
9 2025-04-06 08:57:00     Sunday        True         False


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16316\1998037984.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  upi["is_weekend"] = upi["txn_time"].dt.dayofweek >= 5


In [7]:
upi["time_slot"] = "Night"

upi.loc[
    (upi["hour"] >= 5) & (upi["hour"] <= 11),
    "time_slot"
] = "Morning"

upi.loc[
    (upi["hour"] >= 12) & (upi["hour"] <= 16),
    "time_slot"
] = "Afternoon"

upi.loc[
    (upi["hour"] >= 17) & (upi["hour"] <= 21),
    "time_slot"
] = "Evening"

print(upi[["txn_time", "hour", "time_slot"]].head(10))

             txn_time  hour  time_slot
0 2025-03-26 16:38:00    16  Afternoon
1 2025-04-12 23:47:00    23      Night
2 2025-04-02 14:52:00    14  Afternoon
3 2025-03-14 00:14:00     0      Night
4 2025-06-29 23:38:00    23      Night
5 2025-03-25 05:59:00     5    Morning
6 2025-03-18 13:48:00    13  Afternoon
7 2025-05-25 08:52:00     8    Morning
8 2025-02-08 02:46:00     2      Night
9 2025-04-06 08:57:00     8    Morning


In [8]:
time_slot_count = upi["time_slot"].value_counts()

print("Transaction count per time slot:")
print(time_slot_count)

Transaction count per time slot:
time_slot
Night        875
Morning      869
Afternoon    635
Evening      621
Name: count, dtype: int64


In [9]:
upi = upi.sort_values("txn_time")

upi_indexed = upi.set_index("txn_time")

print(upi_indexed.head())

                     txn_id  amount_inr category  year  month  day    weekday  \
txn_time                                                                        
2025-01-01 02:06:00    1595        3753  Grocery  2025      1    1  Wednesday   
2025-01-01 03:55:00    1429        2161     Fuel  2025      1    1  Wednesday   
2025-01-01 04:41:00    2819        4657     Food  2025      1    1  Wednesday   
2025-01-01 06:44:00    1055        4784     Fuel  2025      1    1  Wednesday   
2025-01-01 08:46:00    2491        3723     Food  2025      1    1  Wednesday   

                     quarter  hour  is_weekend  is_month_end time_slot  
txn_time                                                                
2025-01-01 02:06:00        1     2       False         False     Night  
2025-01-01 03:55:00        1     3       False         False     Night  
2025-01-01 04:41:00        1     4       False         False     Night  
2025-01-01 06:44:00        1     6       False         False   Morn

In [10]:
daily_total = upi_indexed["amount_inr"].resample("D").sum()

print("Daily Transaction Total:")
print(daily_total.head())

daily_total.to_csv("upi_daily.csv")

Daily Transaction Total:
txn_time
2025-01-01    43737
2025-01-02    44902
2025-01-03    30957
2025-01-04    59854
2025-01-05    42050
Freq: D, Name: amount_inr, dtype: int64


In [11]:
weekly_total = upi_indexed["amount_inr"].resample("W").sum()

print("Weekly Transaction Total:")
print(weekly_total.head())

weekly_total.to_csv("upi_weekly.csv")

Weekly Transaction Total:
txn_time
2025-01-05    221500
2025-01-12    254252
2025-01-19    301657
2025-01-26    245905
2025-02-02    284816
Freq: W-SUN, Name: amount_inr, dtype: int64


In [12]:
# Use ME for recent pandas versions
try:
    monthly_total = upi_indexed["amount_inr"].resample("ME").sum()
except ValueError:
    # Use M for older pandas versions
    monthly_total = upi_indexed["amount_inr"].resample("M").sum()

print("Monthly Transaction Total:")
print(monthly_total)

monthly_total.to_csv("upi_monthly.csv")

Monthly Transaction Total:
txn_time
2025-01-31    1236295
2025-02-28    1121615
2025-03-31    1379768
2025-04-30    1278080
2025-05-31    1329934
2025-06-30    1254974
Freq: ME, Name: amount_inr, dtype: int64


In [13]:
daily_sum = daily_total.sum()

weekly_sum = weekly_total.sum()

monthly_sum = monthly_total.sum()

print("Daily Grand Total:", daily_sum)
print("Weekly Grand Total:", weekly_sum)
print("Monthly Grand Total:", monthly_sum)

assert daily_sum == weekly_sum == monthly_sum

print("All three grand totals are equal!")

Daily Grand Total: 7600666
Weekly Grand Total: 7600666
Monthly Grand Total: 7600666
All three grand totals are equal!


In [14]:
# Create a copy of the transaction data
upi_clean = upi.copy()

# Convert UTC time into IST
upi_clean["txn_time_ist"] = (
    upi_clean["txn_time"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Asia/Kolkata")
)

print("First row before conversion:")
print(upi_clean[["txn_time"]].head(1))

print("\nFirst row after conversion:")
print(upi_clean[["txn_time_ist"]].head(1))

First row before conversion:
                txn_time
1594 2025-01-01 02:06:00

First row after conversion:
                  txn_time_ist
1594 2025-01-01 07:36:00+05:30


In [15]:
# Find busiest hour in UTC
busiest_hour_utc = (
    upi_clean["txn_time"].dt.hour.value_counts().idxmax()
)

# Find busiest hour in IST
busiest_hour_ist = (
    upi_clean["txn_time_ist"].dt.hour.value_counts().idxmax()
)

print("Busiest hour in UTC:", busiest_hour_utc)
print("Busiest hour in IST:", busiest_hour_ist)

Busiest hour in UTC: 1
Busiest hour in IST: 20


In [16]:
# Save final cleaned data
upi_clean.to_csv("upi_clean.csv", index=False)

print("upi_clean.csv saved successfully!")

print("\nFinal Data:")
print(upi_clean.head())

print("\nFinal Shape:", upi_clean.shape)

print("\nFinal Data Types:")
print(upi_clean.dtypes)

upi_clean.csv saved successfully!

Final Data:
      txn_id            txn_time  amount_inr category  year  month  day  \
1594    1595 2025-01-01 02:06:00        3753  Grocery  2025      1    1   
1428    1429 2025-01-01 03:55:00        2161     Fuel  2025      1    1   
2818    2819 2025-01-01 04:41:00        4657     Food  2025      1    1   
1054    1055 2025-01-01 06:44:00        4784     Fuel  2025      1    1   
2490    2491 2025-01-01 08:46:00        3723     Food  2025      1    1   

        weekday  quarter  hour  is_weekend  is_month_end time_slot  \
1594  Wednesday        1     2       False         False     Night   
1428  Wednesday        1     3       False         False     Night   
2818  Wednesday        1     4       False         False     Night   
1054  Wednesday        1     6       False         False   Morning   
2490  Wednesday        1     8       False         False   Morning   

                  txn_time_ist  
1594 2025-01-01 07:36:00+05:30  
1428 2025-01-01